# Unsupervised Tree Species Pipeline (Different feature extraction model + Clustering)
End‑to‑end workflow:
1. Crop tree crowns
2. Extract image features
3. PCA dimensionality reduction
4. Multi‑k KMeans clustering
5. Validation using labeled subset
6. t‑SNE visualization
7. Final clustering and pseudo‑labels
8. Attach predictions to polygons
9. Export GeoJSON and KML for visualization

## Import libraries

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import timm

import simplekml

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
from sklearn.manifold import TSNE


## Configuration

In [ ]:
ORTHO_FOLDER = "data/orthomosaics"
POLY_FOLDER = "data/crown_polygons"
LABEL_400_CSV = "data/labels/labeling_sheet.csv"

OUTPUT_ROOT = "outputs_unsupervised"

K_LIST = [2,4,6,8,10,12]
IMG_SIZE = 224
BATCH_SIZE = 32
PCA_COMPONENTS = 50


## Output folders

In [ ]:
DIR_CROWNS = os.path.join(OUTPUT_ROOT,"crowns")
DIR_FEATURES = os.path.join(OUTPUT_ROOT,"features")
DIR_CLUSTER = os.path.join(OUTPUT_ROOT,"clustering")
DIR_VALID = os.path.join(OUTPUT_ROOT,"validation_400")
DIR_PSEUDO = os.path.join(OUTPUT_ROOT,"pseudo_labels")
DIR_GEO = os.path.join(OUTPUT_ROOT,"geo_output")

for d in [DIR_CROWNS,DIR_FEATURES,DIR_CLUSTER,DIR_VALID,DIR_PSEUDO,DIR_GEO]:
    os.makedirs(d,exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",device)


## Step 0 — Crop tree crowns

In [ ]:
orthos = []
for f in os.listdir(ORTHO_FOLDER):
    if f.lower().endswith(".tif"):
        orthos.append((f, rasterio.open(os.path.join(ORTHO_FOLDER,f))))

for gj in os.listdir(POLY_FOLDER):

    if not gj.endswith(".geojson"):
        continue

    prefix = os.path.splitext(gj)[0]
    gdf = gpd.read_file(os.path.join(POLY_FOLDER, gj))

    if "crown_id" in gdf.columns:
        gdf["crown_id"] = gdf["crown_id"].astype(int)
    elif "id" in gdf.columns:
        gdf["crown_id"] = gdf["id"].astype(int)
    else:
        gdf["crown_id"] = gdf.index.astype(int)

    for _,row in tqdm(gdf.iterrows(),total=len(gdf)):

        geom=[row.geometry]
        cid=int(row["crown_id"])
        out_name=f"{prefix}_{cid:03d}.tif"
        out_path=os.path.join(DIR_CROWNS,out_name)

        for _,src in orthos:
            try:
                out_img,out_transform = mask(src,geom,crop=True)

                meta=src.meta.copy()
                meta.update({
                    "height":out_img.shape[1],
                    "width":out_img.shape[2],
                    "transform":out_transform
                })

                with rasterio.open(out_path,"w",**meta) as dst:
                    dst.write(out_img)

                break

            except:
                continue


## Step 1 — Extract DINOv2 features

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485,0.456,0.406),
        std=(0.229,0.224,0.225)
    )
])

model = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=True,
    num_classes=0,
    img_size=224
).to(device)

img_paths = sorted([
    os.path.join(DIR_CROWNS,f)
    for f in os.listdir(DIR_CROWNS)
    if f.endswith(".tif")
])

features=[]
names=[]

for i in tqdm(range(0,len(img_paths),BATCH_SIZE)):

    batch_paths=img_paths[i:i+BATCH_SIZE]
    imgs=[]; nms=[]

    for p in batch_paths:
        imgs.append(transform(Image.open(p).convert("RGB")))
        nms.append(os.path.basename(p))

    batch=torch.stack(imgs).to(device)

    with torch.no_grad():
        feat=model(batch)

    feat=F.normalize(feat,p=2,dim=1)

    features.append(feat.cpu().numpy())
    names.extend(nms)

features=np.vstack(features)

np.save(os.path.join(DIR_FEATURES,"dinov2_features.npy"),features)
pd.DataFrame({"image_name":names}).to_csv(
    os.path.join(DIR_FEATURES,"dinov2_features.csv"),index=False)


## Step 2 — Standardization and PCA

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(features)

pca = PCA(n_components=PCA_COMPONENTS, random_state=42)
X = pca.fit_transform(X)

names_df = pd.DataFrame({"image_name":names})


## Step 3 — Multi‑k clustering evaluation

In [ ]:
labels400 = pd.read_csv(LABEL_400_CSV)
label_map = {"acacia":0,"non_acacia":1}

results=[]
best_k=None
best_acc=-1

for k in K_LIST:

    km = KMeans(n_clusters=k,random_state=42,n_init=10)
    cl = km.fit_predict(X)

    tmp = names_df.copy()
    tmp["cluster"] = cl

    merged = pd.merge(labels400,tmp,on="image_name")
    merged["true"] = merged["label"].map(label_map)

    cluster_to_label={}
    purities=[]

    for c in sorted(merged["cluster"].unique()):

        sub=merged[merged["cluster"]==c]
        counts=sub["true"].value_counts()

        cluster_to_label[c]=counts.idxmax()
        purities.append(counts.max()/len(sub))

    merged["mapped"]=merged["cluster"].map(cluster_to_label)

    acc=(merged["mapped"]==merged["true"]).mean()
    purity=np.mean(purities)

    results.append({"k":k,"purity":purity,"accuracy":acc})

    if acc>best_acc:
        best_acc=acc
        best_k=k
        best_mapping=cluster_to_label

print("Best k:",best_k)


## Step 4 — t‑SNE visualization

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init="pca")

X_tsne = tsne.fit_transform(X)

tsne_df = pd.DataFrame({
    "x":X_tsne[:,0],
    "y":X_tsne[:,1],
    "image_name":names
})


## Step 5 — Final clustering

In [ ]:
km = KMeans(n_clusters=best_k,random_state=42,n_init=10)
final_clusters = km.fit_predict(X)

names_df["cluster"]=final_clusters
names_df["pred"]=names_df["cluster"].map(best_mapping)
names_df["species"]=names_df["pred"].map({0:"acacia",1:"non_acacia"})

names_df.to_csv(os.path.join(DIR_CLUSTER,"final_assignments.csv"),index=False)


## Step 6 — Create pseudo‑labeled dataset

In [ ]:
for sp in ["acacia","non_acacia"]:
    os.makedirs(os.path.join(DIR_PSEUDO,sp),exist_ok=True)

for _,r in names_df.iterrows():

    src=os.path.join(DIR_CROWNS,r["image_name"])
    dst=os.path.join(DIR_PSEUDO,r["species"],r["image_name"])

    shutil.copy(src,dst)


## Step 7 — Attach clustering results to polygons

In [ ]:
all_polys=[]

for gj in os.listdir(POLY_FOLDER):

    if not gj.endswith(".geojson"):
        continue

    prefix=os.path.splitext(gj)[0]
    g=gpd.read_file(os.path.join(POLY_FOLDER,gj))

    if "crown_id" in g.columns:
        g["crown_id"]=g["crown_id"].astype(int)
    elif "id" in g.columns:
        g["crown_id"]=g["id"].astype(int)
    else:
        g["crown_id"]=g.index.astype(int)

    g["image_name"]=g["crown_id"].apply(
        lambda x: f"{prefix}_{int(x):03d}.tif"
    )

    all_polys.append(g)

gdf=pd.concat(all_polys,ignore_index=True)

gdf=gdf.merge(
    names_df[["image_name","species","cluster"]],
    on="image_name",
    how="left"
)


## Step 8 — Export KML for Google Earth

In [ ]:
if gdf.crs is None:
    gdf.set_crs(epsg=32643,inplace=True)

gdf_wgs=gdf.to_crs(epsg=4326)

kml=simplekml.Kml()

for _,r in gdf_wgs.iterrows():

    if pd.isna(r["species"]):
        continue

    geom=r.geometry

    pol=kml.newpolygon(
        name=f"{r['species']} (cluster {r['cluster']})",
        outerboundaryis=list(geom.exterior.coords)
    )

kml.save(os.path.join(DIR_GEO,"species_unsupervised_map.kml"))
print("KML exported")
